In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")
from core.startup import init
engine, memory = init()

In [ ]:
%%writefile /workspace/Projects/cultivated-learning/core/cold_storage.py
import time
import json
import os
from core.memory_store import MemoryUnit, MemoryType


class ColdStorage:
    """Archive for memories that fade below the salience floor.
    Resurfaces archived memories when an unusually strong semantic match occurs."""

    def __init__(self, engine, memory_store, archive_dir, salience_floor=0.15, resurface_threshold=0.75):
        self.engine = engine
        self.memory = memory_store
        self.archive_dir = archive_dir
        self.salience_floor = salience_floor
        self.resurface_threshold = resurface_threshold
        os.makedirs(archive_dir, exist_ok=True)
        self.archive = self._load_archive()
        print(f"Cold storage initialized: {len(self.archive)} archived memories")

    def _load_archive(self):
        """Load archived memories from disk."""
        archive_path = os.path.join(self.archive_dir, "cold_archive.json")
        if os.path.exists(archive_path):
            with open(archive_path, "r") as f:
                return json.load(f)
        return []

    def _save_archive(self):
        """Persist archive to disk."""
        archive_path = os.path.join(self.archive_dir, "cold_archive.json")
        with open(archive_path, "w") as f:
            json.dump(self.archive, f, indent=2)

    def archive_pass(self):
        """Move memories below salience floor to cold storage."""
        all_data = self.memory.collection.get(include=["documents", "metadatas", "embeddings"])
        archived_count = 0

        for i in range(len(all_data["ids"])):
            metadata = all_data["metadatas"][i]
            if metadata["salience_score"] < self.salience_floor:
                # Archive the memory
                entry = {
                    "id": all_data["ids"][i],
                    "document": all_data["documents"][i],
                    "metadata": metadata,
                    "embedding": all_data["embeddings"][i] if all_data["embeddings"] else None,
                    "archived_at": time.time(),
                }
                self.archive.append(entry)

                # Remove from active memory
                self.memory.collection.delete(ids=[all_data["ids"][i]])
                archived_count += 1

        if archived_count > 0:
            self._save_archive()

        print(f"Archive pass: {archived_count} memories moved to cold storage. "
              f"Total archived: {len(self.archive)}")
        return archived_count

    def resurface(self, query_text):
        """Check cold storage for unusually strong matches to the query."""
        if not self.archive:
            return []

        query_embedding = self.engine.get_embedding(query_text)
        resurfaced = []

        for entry in self.archive:
            if entry["embedding"] is None:
                continue

            # Calculate cosine similarity
            import numpy as np
            query_vec = np.array(query_embedding)
            stored_vec = np.array(entry["embedding"])
            similarity = np.dot(query_vec, stored_vec) / (
                np.linalg.norm(query_vec) * np.linalg.norm(stored_vec)
            )

            if similarity >= self.resurface_threshold:
                # Restore to active memory with boosted salience
                mem = MemoryUnit.from_chroma(
                    id=entry["id"],
                    document=entry["document"],
                    metadata=entry["metadata"],
                )
                mem.salience_score = 0.5  # Reset to neutral
                mem.last_accessed = time.time()
                mem.tags.append("resurfaced")
                self.memory.store(mem)
                resurfaced.append(mem)

        # Remove resurfaced memories from archive
        if resurfaced:
            resurfaced_ids = {m.id for m in resurfaced}
            self.archive = [e for e in self.archive if e["id"] not in resurfaced_ids]
            self._save_archive()
            print(f"Resurfaced {len(resurfaced)} memories from cold storage!")

        return resurfaced

    def get_stats(self):
        return {
            "archived_count": len(self.archive),
            "salience_floor": self.salience_floor,
            "resurface_threshold": self.resurface_threshold,
        }

In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")

from engine.inference import InferenceEngine
from core.memory_store import MemoryStore
from core.cold_storage import ColdStorage

engine = InferenceEngine("/workspace/models/results/Mistral-7B-Instruct-v0.3")
engine.load()

memory = MemoryStore(
    persist_dir="/workspace/Projects/cultivated-learning/data/memory_db",
    engine=engine
)

cold = ColdStorage(
    engine=engine,
    memory_store=memory,
    archive_dir="/workspace/Projects/cultivated-learning/data/cold_storage"
)

print("\nRunning archive pass...")
cold.archive_pass()

print("\nTesting resurface...")
resurfaced = cold.resurface("What is the user's name?")
print(f"Resurfaced: {len(resurfaced)}")

print(f"\nCold storage stats: {cold.get_stats()}")